# 1.3 - Regresión lineal múltiple y gradiente descendente: calorías

* En este notebook vamos a estudiar la **regresión lineal múltiple** y cómo ajustar sus parámetros mediante **gradiente descendente**.
* Resolveremos un problema de regresión: estimar las **calorías quemadas** a partir del tiempo de carrera, el peso y la velocidad.
* En la parte interactiva veremos cómo cambia el modelo al ejecutar **un epoch cada vez**.

---
# 1. Regresión lineal múltiple

* En la regresión lineal simple utilizamos una variable de entrada. En la **regresión lineal múltiple** utilizamos dos o más variables para predecir una salida numérica.
* La forma general del **modelo** con $n$ variables de entrada es:

$$\hat{y}=\beta_0+\beta_1 x_1+\beta_2 x_2+\cdots+\beta_n x_n$$

* $\beta_0$ es el **término independiente**. Cada $\beta_j$ multiplica a una variable de entrada y representa su contribución al modelo lineal, manteniendo las demás constantes.
* El modelo define la **forma de la predicción**. Sus parámetros $\beta_0,\beta_1,\ldots,\beta_n$ se ajustan durante el entrenamiento.

---
# 2. Modelo para estimar las calorías

* En este ejemplo tenemos tres variables de entrada: **tiempo**, **peso** y **velocidad**. La salida es la cantidad de **calorías**.
* Definimos la forma del modelo:

$$\widehat{\text{calorías}} = \beta_0 + \beta_1 \cdot \text{tiempo} + \beta_2 \cdot \text{peso} + \beta_3 \cdot \text{velocidad}$$

* $\beta_1$ corresponde al tiempo, $\beta_2$ al peso y $\beta_3$ a la velocidad. El modelo tiene **cuatro parámetros**, incluido $\beta_0$.
* Antes del entrenamiento conocemos esta forma, pero no los valores que mejor se ajustan a los datos. Usaremos el **optimizador de gradiente descendente** para encontrarlos reduciendo la pérdida.

> La distinción es la misma en una red neuronal: la arquitectura define la forma del modelo; un optimizador, como gradiente descendente, Adam o RMSprop, actualiza sus pesos y sesgos para reducir la pérdida.

---
# 3. Función de pérdida y métricas

* Para cada observación calculamos el error como la diferencia entre la predicción y el valor real:

$$e^{(i)} = \hat{y}^{(i)} - y^{(i)}$$

* Utilizamos el **MSE** como función de pérdida. También mostramos el **MAE**, que expresa el error medio en las mismas unidades que las calorías:

$$E(\beta_0,\ldots,\beta_n)=MSE=\frac{1}{N}\sum_{i=1}^{N}(\hat{y}^{(i)}-y^{(i)})^2$$

$$MAE=\frac{1}{N}\sum_{i=1}^{N}|\hat{y}^{(i)}-y^{(i)}|$$

* El MSE eleva los errores al cuadrado y da más peso a los errores grandes. El optimizador minimiza el MSE; el MAE se muestra como métrica adicional.

---
# 4. Gradiente descendente

* Partimos de unos valores iniciales para los parámetros. Calculamos las predicciones y sus errores para obtener el gradiente de la función de pérdida.
* Para una variable $x_j$, las derivadas parciales del MSE son:

$$\frac{\partial E}{\partial \beta_0}=\frac{2}{N}\sum_{i=1}^{N}e^{(i)}$$

$$\frac{\partial E}{\partial \beta_j}=\frac{2}{N}\sum_{i=1}^{N}e^{(i)}x_j^{(i)} \quad (j=1,\ldots,n)$$

* Actualizamos **todos los parámetros a la vez**, utilizando los valores anteriores:

$$\beta_j^{new}:=\beta_j^{old}-\alpha\frac{\partial E}{\partial\beta_j} \quad (j=0,\ldots,n)$$

* Sustituyendo el error $e^{(i)} = f(x^{(i)}) - y^{(i)}$, las actualizaciones se desarrollan así:

$$\beta_0^{new} := \beta_0^{old} - \alpha\,\frac{2}{N}\sum_{i=1}^{N}(f(x^{(i)}) - y^{(i)})$$

$$\beta_1^{new} := \beta_1^{old} - \alpha\,\frac{2}{N}\sum_{i=1}^{N}(f(x^{(i)}) - y^{(i)})\,x_1^{(i)}$$

$$\vdots$$

$$\beta_n^{new} := \beta_n^{old} - \alpha\,\frac{2}{N}\sum_{i=1}^{N}(f(x^{(i)}) - y^{(i)})\,x_n^{(i)}$$

* Cada gradiente de $\beta_j$ se multiplica por su variable $x_j$. Para $\beta_0$ no aparece ninguna variable: equivale a multiplicar por 1. El factor 2 aparece al derivar el **MSE**, que es la pérdida elegida.

* $\alpha$ es el **learning rate** o tasa de aprendizaje. Si es demasiado pequeño el ajuste puede ser lento; si es demasiado grande, la pérdida puede aumentar o divergir.
* En este ejemplo utilizamos **batch gradient descent**: cada epoch recorre todas las observaciones y realiza una actualización de los cuatro parámetros.

### Pseudocódigo

```text
Inicializar beta_0, beta_1, beta_2 y beta_3 a cero
Para cada epoch:
    Calcular las predicciones y los errores de todos los datos
    Calcular las derivadas parciales de todos los parámetros
    Actualizar los cuatro parámetros simultáneamente
    Calcular MAE y MSE del modelo actualizado
```

---
# 5. Conjunto de datos

* Cargamos `./datasets/calories_time_weight_speed.csv`.
* Usamos las columnas **Tiempo**, **Peso** y **Velocidad**, en ese orden, como variables de entrada y **Calorias** como salida.
* Mostramos las cinco primeras observaciones. En cada epoch utilizaremos el conjunto completo.

In [1]:
from io import BytesIO
import base64

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

dataset = pd.read_csv('./datasets/calories_time_weight_speed.csv')
X = dataset[['Tiempo', 'Peso', 'Velocidad']].to_numpy(dtype=float)
y = dataset['Calorias'].to_numpy(dtype=float)

display(dataset[['Tiempo', 'Peso', 'Velocidad', 'Calorias']].head())
print(f'Número de observaciones: {len(dataset)}')

,Tiempo,Peso,Velocidad,Calorias
0,13.7,61.2,12.3,177.9
1,51.9,62.6,11.5,644.1
2,55.4,69.3,14.0,929.4
3,61.9,80.1,11.0,941.2
4,19.9,95.6,9.2,303.1


Número de observaciones: 5000


---
# 6. Implementación de un epoch

* `un_epoch` calcula el error con los parámetros anteriores y actualiza a la vez $\beta_0$ y las tres betas.
* Las métricas se calculan después con el **modelo actualizado**, que es el que se muestra en cada fila de la tabla.

In [2]:
def un_epoch(X, y, beta_0, betas, alpha):
    """Realiza una actualización simultánea por lote minimizando el MSE."""
    predicciones = beta_0 + X @ betas
    errores = predicciones - y

    gradiente_beta_0 = 2 * np.mean(errores)
    gradiente_betas = 2 * np.mean(errores[:, np.newaxis] * X, axis=0)

    nuevo_beta_0 = beta_0 - alpha * gradiente_beta_0
    nuevas_betas = betas - alpha * gradiente_betas
    return nuevo_beta_0, nuevas_betas


def calcular_metricas(X, y, beta_0, betas):
    """Calcula MAE y MSE del modelo indicado."""
    errores = beta_0 + X @ betas - y
    return np.mean(np.abs(errores)), np.mean(errores ** 2)

---
# 7. Entrenamiento interactivo

* Ejecuta las celdas en orden. La primera fila, **epoch 0**, muestra el modelo inicial con **todas las betas a cero** y sus métricas.
* Indica cuántos epochs quieres ejecutar (por defecto, **5**) y pulsa **«Ejecutar»**. Se añade una sola fila con el modelo del último epoch ejecutado. Por ejemplo, con 5 epochs por pulsación, la tabla muestra los epochs 0, 5, 10, 15... La gráfica muestra todos los epochs intermedios. MSE y MAE aparecen también junto a α.
* La gráfica utiliza un eje vertical para cada métrica, porque el MSE y el MAE tienen unidades y escalas diferentes.
* Puedes modificar **α** para la siguiente actualización. La tasa inicial es pequeña porque las variables tienen escalas distintas; si se elige una tasa demasiado alta, el error puede aumentar.
* **«Reiniciar»** devuelve las betas a cero y conserva la tasa de aprendizaje y el número de epochs elegidos.
* En la zona de **inferencia**, introduce tiempo, peso y velocidad para obtener la predicción con el último modelo. El cálculo utiliza los valores completos de los parámetros, aunque la tabla los muestre con dos decimales.

> Las métricas se calculan sobre los datos de entrenamiento. Para evaluar la generalización se necesita un conjunto de evaluación independiente.

In [3]:
# Configuración inicial: parámetros a cero.
ALPHA_INICIAL = 0.00005
estado = {'epoch': 0, 'beta_0': 0.0, 'betas': np.zeros(3), 'historial': [], 'historial_grafica': []}

boton_epoch = widgets.Button(description='Ejecutar', button_style='primary',
                             icon='step-forward', layout=widgets.Layout(width='130px'))
numero_epochs = widgets.IntText(value=5, description='Epochs:',
                                layout=widgets.Layout(width='150px'),
                                style={'description_width': '65px'})
boton_reiniciar = widgets.Button(description='Reiniciar', icon='refresh',
                                 layout=widgets.Layout(width='130px'))
alpha = widgets.FloatText(value=ALPHA_INICIAL, description='α:', step=0.00001,
                          layout=widgets.Layout(width='180px'),
                          style={'description_width': '30px'})
alpha.add_class('tasa-aprendizaje-multiple')
estilo = widgets.HTML('<style>.tasa-aprendizaje-multiple input, .tasa-aprendizaje-multiple label {font-size: 19px !important;}</style>')
resumen = widgets.HTML()
salida_tabla = widgets.HTML(layout=widgets.Layout(width='100%'))
salida_grafica = widgets.HTML(layout=widgets.Layout(width='100%'))
panel_tabla = widgets.Box([salida_tabla], layout=widgets.Layout(width='58%', min_width='0px', max_height='460px', overflow='auto'))

# Datos de entrada para utilizar el último modelo entrenado (inferencia).
tiempo = widgets.FloatText(value=30.0, description='Tiempo (min):',
                           layout=widgets.Layout(width='190px'), style={'description_width': '100px'})
peso = widgets.FloatText(value=80.0, description='Peso (kg):',
                         layout=widgets.Layout(width='165px'), style={'description_width': '85px'})
velocidad = widgets.FloatText(value=12.0, description='Velocidad (km/h):',
                              layout=widgets.Layout(width='230px'), style={'description_width': '135px'})
prediccion = widgets.HTML(layout=widgets.Layout(width='45%', min_width='0px'))
for entrada in (tiempo, peso, velocidad):
    entrada.layout.width = '100%'
campos_inferencia = widgets.VBox([tiempo, peso, velocidad], layout=widgets.Layout(width='55%', min_width='0px'))
zona_inferencia = widgets.HBox([campos_inferencia, prediccion],
                               layout=widgets.Layout(width='100%', align_items='center'))
panel_grafica = widgets.VBox([salida_grafica, zona_inferencia],
                             layout=widgets.Layout(width='42%', min_width='0px'))


def registrar_modelo(mostrar_en_tabla=True):
    beta_0, betas = estado['beta_0'], estado['betas']
    mae, mse = calcular_metricas(X, y, beta_0, betas)
    estado['historial_grafica'].append({'Epoch': estado['epoch'], 'MSE': mse, 'MAE': mae})
    if mostrar_en_tabla:
        modelo = (f"ŷ = {beta_0:.2f} + {betas[0]:.2f} · tiempo "
                  f"+ {betas[1]:.2f} · peso + {betas[2]:.2f} · velocidad")
        estado['historial'].append({'Epoch': estado['epoch'], 'Modelo': modelo,
                                    'MSE': mse, 'MAE': mae})


def actualizar_prediccion(_=None):
    entradas = np.array([tiempo.value, peso.value, velocidad.value])
    if not np.isfinite(entradas).all():
        prediccion.value = '<b style="color: #c00; font-size: 18px">Introduce valores finitos.</b>'
        return
    calorias = estado['beta_0'] + entradas @ estado['betas']
    prediccion.value = f"<div style='font-size: 19px'><b>= {calorias:.2f} calorías quemadas (estimadas)</b></div>"


def actualizar_vista():
    ultimo = estado['historial'][-1]
    resumen.value = (f"<div style='font-size: 19px; line-height: 1.8'>"
                     f"<b>Epoch:</b> {estado['epoch']} &nbsp;&nbsp; "
                     f"<b>MSE:</b> {ultimo['MSE']:.2f} &nbsp;&nbsp; "
                     f"<b>MAE:</b> {ultimo['MAE']:.2f}</div>")
    tabla = pd.DataFrame(estado['historial'])
    datos_grafica = pd.DataFrame(estado['historial_grafica'])
    # Actualizamos la tabla sin limpiar otras salidas del notebook.
    with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
        salida_tabla.value = tabla.to_html(index=False, border=0,
                                          formatters={'MSE': '{:.2f}'.format, 'MAE': '{:.2f}'.format})
    # Actualizamos una imagen persistente: evita que la gráfica desaparezca
    # por las actualizaciones del área de salida del notebook.
    fig, ax_mse = plt.subplots(figsize=(5.5, 4))
    ax_mae = ax_mse.twinx()
    linea_mse, = ax_mse.plot(datos_grafica['Epoch'], datos_grafica['MSE'], 'o-', color='darkorange', label='MSE')
    linea_mae, = ax_mae.plot(datos_grafica['Epoch'], datos_grafica['MAE'], 'o-', color='seagreen', label='MAE')
    ax_mse.set_xlabel('Epoch')
    ax_mse.set_ylabel('MSE (calorías²)', color='darkorange')
    ax_mae.set_ylabel('MAE (calorías)', color='seagreen')
    ax_mse.set_title('Evolución del error')
    ax_mse.grid(alpha=0.3)
    ax_mse.legend(handles=[linea_mse, linea_mae], loc='best')
    fig.tight_layout()
    with BytesIO() as imagen:
        fig.savefig(imagen, format='png', dpi=120, bbox_inches='tight')
        imagen_base64 = base64.b64encode(imagen.getvalue()).decode('ascii')
        salida_grafica.value = (f'<img src="data:image/png;base64,{imagen_base64}" '
                                'alt="Evolución del MSE y MAE" style="width:100%; height:auto; display:block;">')
    plt.close(fig)
    actualizar_prediccion()


def ejecutar_epoch(_):
    if not np.isfinite(alpha.value) or alpha.value <= 0:
        resumen.value = '<b style="color: #c00; font-size: 19px">Introduce un valor de α mayor que cero y finito.</b>'
        return
    if numero_epochs.value < 1:
        resumen.value = '<b style="color: #c00; font-size: 19px">Introduce un número de epochs mayor que cero.</b>'
        return
    for indice in range(numero_epochs.value):
        estado['beta_0'], estado['betas'] = un_epoch(
            X, y, estado['beta_0'], estado['betas'], alpha.value
        )
        estado['epoch'] += 1
        registrar_modelo(mostrar_en_tabla=(indice == numero_epochs.value - 1))
    actualizar_vista()


def reiniciar(_=None):
    estado.update(epoch=0, beta_0=0.0, betas=np.zeros(3), historial=[], historial_grafica=[])
    registrar_modelo()
    actualizar_vista()


boton_epoch.on_click(ejecutar_epoch)
boton_reiniciar.on_click(reiniciar)
alpha.observe(lambda cambio: actualizar_vista(), names='value')
for entrada in (tiempo, peso, velocidad):
    entrada.observe(actualizar_prediccion, names='value')
display(estilo,
        widgets.HBox([boton_epoch, numero_epochs, boton_reiniciar, alpha, resumen]),
        widgets.HBox([panel_tabla, panel_grafica], layout=widgets.Layout(width='100%', align_items='flex-start')),
       )
reiniciar()

HTML(value='<style>.tasa-aprendizaje-multiple input, .tasa-aprendizaje-multiple label {font-size: 19px !import…